# 13 — Product Search Evaluation

Reranker برای هر query یک بار اجرا و cache می‌شود؛ tuning وزن‌ها بعدش offline است. بهترین policy فقط روی DEV انتخاب و روی TEST گزارش می‌شود.

In [1]:
import os
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.rag.config import load_config
from src.rag.evaluation.product_search_runtime import load_product_search_evaluation_context
from src.rag.evaluation.product_search_runner import ProductSearchEvaluationRunner
from src.rag.evaluation.product_search_qrels import resolve_qrels
from src.rag.evaluation.product_search_metrics import (
    evaluate_policies, aggregate_metrics, select_best_policy, candidate_recall, failure_analysis
)

In [2]:
cfg = load_config(PROJECT_ROOT / "configs" / "product_search_evaluation.yaml")["product_search_evaluation"]
EVAL_DIR = PROJECT_ROOT / "data" / "evaluation" / "product_search"
queries = pd.read_csv(EVAL_DIR / "queries.csv")
qrels_raw = pd.read_parquet(EVAL_DIR / "qrels.parquet")
qrels = resolve_qrels(qrels_raw, allow_teacher_proxy=cfg["allow_teacher_proxy"])
display(qrels["label_source"].value_counts())

label_source
llm_teacher_proxy    587
Name: count, dtype: int64

In [3]:
context = load_product_search_evaluation_context(
    project_root=PROJECT_ROOT,
    api_key=os.environ["METIS_API_KEY"],
    base_url=os.environ["METIS_BASE_URL"],
)
runner = ProductSearchEvaluationRunner(
    metadata_retriever=context.metadata,
    review_evidence=context.review_evidence,
    reranker=context.reranker,
    metadata_k=cfg["metadata_k"],
    reranker_k=cfg["reranker_k"],
)
result = runner.run(
    queries=queries,
    candidates_path=EVAL_DIR / "evaluation_candidates.parquet",
    telemetry_path=EVAL_DIR / "evaluation_telemetry.csv",
    resume=True,
)
candidates = result["candidates"]
telemetry = result["telemetry"]
print("Queries executed:", candidates["query_id"].nunique())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Queries executed: 30


In [4]:
per_query = evaluate_policies(
    candidates=candidates,
    qrels=qrels,
    policies=cfg["policies"],
    ks=tuple(cfg["metrics_ks"]),
    relevant_threshold=cfg["relevant_grade_threshold"],
)
aggregate = aggregate_metrics(per_query)
display(aggregate[aggregate["split"] == "dev"].sort_values("ndcg@10", ascending=False))
selected_policy_name = select_best_policy(aggregate, metric="ndcg@10")
print("Selected on DEV:", selected_policy_name)

,split,policy,mrr@10,precision@1,hit_rate@1,pool_recall@1,ndcg@1,precision@3,hit_rate@3,pool_recall@3,ndcg@3,precision@5,hit_rate@5,pool_recall@5,ndcg@5,precision@10,hit_rate@10,pool_recall@10,ndcg@10
0,dev,llm_only,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
2,dev,tiered_30_70,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
3,dev,weighted_20_80,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
5,dev,weighted_40_60,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
4,dev,weighted_30_70,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
6,dev,weighted_50_50,0.625000,0.60,0.60,0.077499,0.559524,0.6,0.65,0.278925,0.585524,0.53,0.65,0.367614,0.571341,0.380,0.65,0.463438,0.535760
1,dev,metadata_only,0.519643,0.45,0.45,0.046986,0.445238,0.4,0.55,0.167319,0.430351,0.41,0.60,0.259059,0.446348,0.355,0.65,0.426949,0.453358


Selected on DEV: llm_only


In [5]:
test_results = aggregate[aggregate["split"] == "test"].sort_values("ndcg@10", ascending=False)
display(test_results)
print("FINAL policy:", selected_policy_name)
display(test_results[test_results["policy"] == selected_policy_name])

,split,policy,mrr@10,precision@1,hit_rate@1,pool_recall@1,ndcg@1,precision@3,hit_rate@3,pool_recall@3,ndcg@3,precision@5,hit_rate@5,pool_recall@5,ndcg@5,precision@10,hit_rate@10,pool_recall@10,ndcg@10
7,test,llm_only,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
9,test,tiered_30_70,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
10,test,weighted_20_80,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
12,test,weighted_40_60,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
11,test,weighted_30_70,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
13,test,weighted_50_50,0.700000,0.7,0.7,0.141026,0.700000,0.633333,0.7,0.365385,0.676536,0.60,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651
8,test,metadata_only,0.484286,0.4,0.4,0.053526,0.342857,0.300000,0.5,0.123718,0.334011,0.34,0.6,0.264744,0.366146,0.36,0.7,0.591987,0.473186


FINAL policy: llm_only


,split,policy,mrr@10,precision@1,hit_rate@1,pool_recall@1,ndcg@1,precision@3,hit_rate@3,pool_recall@3,ndcg@3,precision@5,hit_rate@5,pool_recall@5,ndcg@5,precision@10,hit_rate@10,pool_recall@10,ndcg@10
7,test,llm_only,0.7,0.7,0.7,0.141026,0.7,0.633333,0.7,0.365385,0.676536,0.6,0.7,0.522436,0.666685,0.38,0.7,0.625962,0.637651


In [6]:
recall_table = candidate_recall(
    candidates, qrels,
    relevant_threshold=cfg["relevant_grade_threshold"],
    shortlist_k=cfg["reranker_k"],
)
display(recall_table.groupby("split")[["candidate_recall@12", "candidate_hit@12"]].mean())

selected_policy = next(policy for policy in cfg["policies"] if policy["name"] == selected_policy_name)
failures = failure_analysis(
    candidates, qrels, selected_policy,
    relevant_threshold=cfg["relevant_grade_threshold"],
    shortlist_k=cfg["reranker_k"],
)
display(failures["failure_type"].value_counts())
display(failures[failures["failure_type"] != "ok"])

,candidate_recall@12,candidate_hit@12
split,,
dev,0.487544,0.65
test,0.633654,0.70


failure_type
ok                            19
no_relevant_in_judged_pool     8
candidate_retrieval_miss       2
top_rank_error                 1
Name: count, dtype: int64

,query_id,query_type,split,query,failure_type,top1_id,top1_grade,top1_title
3,q004,brand,dev,پاوربانک شیائومی 20000,no_relevant_in_judged_pool,104647,0,ترازو هوشمند شیائومی مدل Mi
4,q005,brand,test,هندزفری بی سیم انکر,no_relevant_in_judged_pool,12089628,0,میکروفن بی سیم آر اند بی مدل RB-1
8,q009,attribute,test,سرخ کن بدون روغن با ظرفیت حداقل 6 لیتر,no_relevant_in_judged_pool,12433908,0,روغن گل سرخ بارجین مدل 06 حجم 120 میلی لیتر
9,q010,attribute,test,ماوس بی سیم ارگونومیک,no_relevant_in_judged_pool,12089628,0,میکروفن بی سیم آر اند بی مدل RB-1
10,q011,attribute,dev,کوله لپ تاپ 15.6 اینچ ضد آب,top_rank_error,397150,1,کوله پشتی لپ تاپ های سیرا مدل Brody مناسب برای...
13,q014,experiential,dev,هدفون راحت برای استفاده طولانی,candidate_retrieval_miss,4314827,0,کتاب اسرار طولانی مدت برای معاملات کوتاه مدت ا...
14,q015,experiential,dev,جارو شارژی سبک با مکش خوب,no_relevant_in_judged_pool,11005259,0,جاروبرقی اسباب بازی طرح جارو شارژی کد V590GP
15,q016,experiential,dev,اسپیکر با صدای شفاف و بیس خوب,candidate_retrieval_miss,11546051,0,آینه جیبی خندالو مدل صدا کن مرا صدای تو خوب اس...
22,q023,negative,dev,پاوربانک که داغ نکنه,no_relevant_in_judged_pool,11987778,0,شیکر میو مدل پاوربانک گنجایش 0.38 لیتر
26,q027,multi_constraint,dev,هندزفری بلوتوث با باتری خوب و میکروفون مناسب تماس,no_relevant_in_judged_pool,161661,0,میکروفون حرفه ای زوم مدل IQ6 مناسب برای آیفون


In [7]:
print("Mean latency sec:", round(telemetry["total_latency_ms"].mean() / 1000, 2))
print("P95 latency sec:", round(telemetry["total_latency_ms"].quantile(0.95) / 1000, 2))
if "reranker_total_tokens" in telemetry:
    print("Total reranker tokens:", int(telemetry["reranker_total_tokens"].sum()))
if "reranker_estimated_cost_usd" in telemetry and telemetry["reranker_estimated_cost_usd"].notna().any():
    print("Estimated reranker cost USD:", float(telemetry["reranker_estimated_cost_usd"].sum()))

per_query.to_csv(EVAL_DIR / "per_query_metrics.csv", index=False)
aggregate.to_csv(EVAL_DIR / "aggregate_metrics.csv", index=False)
recall_table.to_csv(EVAL_DIR / "candidate_recall.csv", index=False)
failures.to_csv(EVAL_DIR / "failure_analysis.csv", index=False)

Mean latency sec: 12.1
P95 latency sec: 19.06
Total reranker tokens: 76932
